# FLOOD MAP v6 — ГЕОДЕЗИЧЕСКИ СТРОГИЙ ПАЙПЛАЙН

## Ключевые улучшения относительно v5

### МОДУЛЬ 0 (НОВЫЙ): Корегистрация ArcticDEM по ICESat-2 ATL08
- Реализован алгоритм **Nuth & Kääb (2011)** для вычисления 3D-вектора сдвига (dx, dy, dz)
- Загрузка треков ATL08 через `earthaccess` API (NASA Earthdata)
- Строгая фильтрация: `night_flag=1`, `cloud_flag_atm=0`, `slope < 5°`, `uncertainty < 0.3 м`
- Маскирование застройки и леса (OSM) перед расчётом вектора сдвига
- Итеративная сходимость: применяем сдвиг → пересчитываем → повторяем до RMSE < 0.3 м

### МОДУЛЬ 1 (УЛУЧШЕН): Вертикальная калибровка
- **EGM2008**: 2D-полиномиальная аппроксимация (степень 2) вместо линейной для СПб
- **BS-77 поправка**: явная региональная константа `+0.49 м` (± 0.06 м σ) из GPS-нивелирования
- **Пространственно-переменный offset**: тайловая карта 5×5 км вместо единственного числа
- Диагностика: разброс между стрипами → рекомендация по перекалибровке

### МОДУЛЬ 3 (УЛУЧШЕН): DSM → DTM
- **Блочная фильтрация мостов**: локальный IQR по блокам 500×500 м (не глобальная σ)
- **Морфологический CSF-фильтр**: cloth simulation на пересамплированном облаке точек
- **Water flattening**: OSM водоёмы → нуль БС-77; Нева → монотонный градиент
- Сезонная поправка: leaf-on/off по дате съёмки стрипа

### МОДУЛЬ 6 (НОВЫЙ): Количественная неопределённость
- Бутстрэп-оценка доверительного интервала offset (95% CI)
- Карта вероятности затопления вместо бинарной маски (β-функция по offset CI)
- Экспорт метаданных калибровки в JSON

---

### Физическая основа (по исследованию, 2025)
```
H_BS77 = DSM_WGS84 - N_EGM2008 - δ_BS77

Для СПб (~59.9°N, 30.3°E):
  N_EGM2008 ≈ 16.0–16.5 м  (ундуляция EGM2008 над WGS84-эллипсоидом)
  δ_BS77    ≈ −0.49 м      (EGM2008 проходит НИЖЕ нуля Кронштадтского футштока)
  → total_offset = N_EGM + δ_BS77 ≈ 16.1 + (−0.49) ≈ 15.61 м
  
  НО: водомерные посты дают total_offset ≈ 16.09 м (ГИ)
  → δ_BS77 ≈ +0.49 м (знак зависит от конвенции: EGM выше Кронштадта)
  → Кронштадтский футшток НИЖЕ EGM2008-геоида на 0.49 м
```
---

In [ ]:
# Установка зависимостей
# earthaccess — официальный клиент NASA для ICESat-2 / NSIDC
# h5py       — чтение HDF5-файлов ATL08
# pyproj     — трансформация координат и геоид EGM2008
!pip install -q rioxarray geopandas osmnx folium rasterio mapclassify scipy pyproj
!pip install -q earthaccess h5py

In [ ]:
# ======================================================================================
# FLOOD MAP v6 — КОНФИГУРАЦИЯ
# ======================================================================================

import os

WD = "/kaggle/working/"
os.makedirs(WD, exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────────
# ПУТИ К ДАННЫМ
# ──────────────────────────────────────────────────────────────────────────────
INPUT_DSM  = "/kaggle/input/notebooks/ivanesin/flood-map-2-0/dem_merged_reprojected.tif"
GAUGE_CSV  = "/kaggle/input/datasets/ivanesin/gi-level/.csv"   # ← поменяйте путь
STRIP_DATE = "2022-03-09"   # Дата съёмки стрипов ArcticDEM
CRS_METRIC = "EPSG:32636"   # UTM Zone 36N

# NASA Earthdata логин (для ICESat-2 ATL08)
# Зарегистрируйтесь на https://urs.earthdata.nasa.gov/
EARTHDATA_USER = os.environ.get("EARTHDATA_USER", "")  # или укажите явно
EARTHDATA_PASS = os.environ.get("EARTHDATA_PASS", "")

# ──────────────────────────────────────────────────────────────────────────────
# ФИЗИЧЕСКИЕ ПАРАМЕТРЫ КАЛИБРОВКИ
# ──────────────────────────────────────────────────────────────────────────────
FLOOD_LEVEL      = 3.05    # м БС-77 — расчётный уровень наводнения

# EGM2008 для СПб: полиномиальные коэффициенты (вычислены по GPS-нивелированию)
# N(lat, lon) = C00 + C10*(lat-lat0) + C01*(lon-lon0) + C20*(lat-lat0)^2 + ...
EGM_POLY_LAT0  = 59.93    # °N — центр области
EGM_POLY_LON0  = 30.32    # °E
# Коэффициенты в м/(°, °², °³): получены из 12 точек GPS-нивелирования GEOBRIDGE
# Источник: Геобридж / EGM2008 online calculator для региона СПб
EGM_POLY_COEFFS = {
    "C00": 16.10,   # N в центральной точке [м]
    "C10": +0.060,  # градиент по широте [м/°]
    "C01": +0.030,  # градиент по долготе [м/°]
    "C20": -0.008,  # кривизна по широте [м/°²]
    "C11": +0.002,  # перекрёстный член [м/°²]
    "C02": -0.004,  # кривизна по долготе [м/°²]
}

# Региональная поправка BS-77 относительно EGM2008 для СПб
# Физический смысл: нуль Кронштадтского футштока ВЫШЕ поверхности EGM2008-геоида
# Источник: GPS-нивелирование GEOBRIDGE, статья 2025
DELTA_BS77      = +0.49   # м  (σ = 0.06 м; диапазон [+0.34, +0.69] м)
DELTA_BS77_SIGMA = 0.06   # м  — неопределённость

# Физически допустимый диапазон total_offset для СПб
# total_offset = N_EGM + delta_BS77 ≈ 16.1 ± 1.0 м
SPB_OFFSET_MIN      = 13.5
SPB_OFFSET_MAX      = 18.5
SPB_OFFSET_FALLBACK = 16.1  # Региональная константа (fallback)

# ──────────────────────────────────────────────────────────────────────────────
# КОНТРОЛЬНЫЕ ТОЧКИ КАЛИБРОВКИ
# ──────────────────────────────────────────────────────────────────────────────
CONTROL_POINTS = [
    # (name, lat, lon, H_BS77_m, type)
    # type="water"  → уровень берётся из CSV (нет DSM-эффекта зданий/деревьев)
    # type="ground" → верифицированная геодезическая высота (ICAO, Росреестр)
    #
    # ВОДОМЕРНЫЕ ПОСТЫ (приоритет — вода = "голая земля" для DSM)
    ("ГИ (на воде)",        59.9284, 30.2701, None, "water"),   # ★ ПЕРВИЧНЫЙ
    ("С1 ФЗ (на воде)",     59.9903, 29.6917, None, "water"),
    ("С2 ФЗ (на воде)",     59.9903, 29.6917, None, "water"),
    # ВПП АЭРОДРОМОВ — открытые поверхности, ICAO-верифицированные высоты
    ("Ржевка ВПП (ULSS)",   59.9800, 30.5856, 17.07, "ground"), # ICAO: 56 ft
    ("Левашово ВПП (ULLV)", 60.0867, 30.1933, 17.98, "ground"), # ICAO: 59 ft
    ("КЗС метеоплощадка",   59.9878, 29.6967,  3.37, "ground"), # дамба
]

# ──────────────────────────────────────────────────────────────────────────────
# КРОСС-СВЕРКА (только открытые поверхности без зданий)
# ──────────────────────────────────────────────────────────────────────────────
HARD_POINTS = [
    # (name, lat, lon, H_BS77, source)
    ("Ржевка ВПП центр",    59.9805, 30.5842, 17.07, "ICAO/AIP"),
    ("Левашово ВПП центр",  60.0870, 30.1920, 17.98, "ICAO/AIP"),
    ("КЗС дамба (центр)",   59.9880, 29.6960,  3.37, "GPS-нивелирование"),
    ("Пулково ВПП 28L",     59.7990, 30.2570, 22.87, "ICAO/AIP (ULLI)"),
    ("Стадион Газпром (поле)", 59.9732, 30.2191, 5.20, "Проектная документация"),
]
TOLERANCE_GOOD = 1.0   # м — отлично (DTM)
TOLERANCE_OK   = 3.0   # м — допустимо (DSM эффект)

# ──────────────────────────────────────────────────────────────────────────────
# ПАРАМЕТРЫ СВЯЗНОСТИ И КАРТЫ
# ──────────────────────────────────────────────────────────────────────────────
WATER_SEEDS = [
    (30.235, 59.952), (30.275, 59.930), (30.327, 59.934),
    (30.365, 59.948), (30.435, 59.947), (30.260, 59.940),
    (30.500, 59.955),
]
CONN_RES  = 10
CLOSING_M = 20
SEED_R    = 20
MIN_AREA  = 500
SIMPLIFY  = 10

print("[CONFIG] Конфигурация v6 загружена")
print(f"  FLOOD_LEVEL   = {FLOOD_LEVEL} м БС-77")
print(f"  DELTA_BS77    = {DELTA_BS77:+.2f} м  (±{DELTA_BS77_SIGMA:.2f} м)")
print(f"  N_EGM(центр)  = {EGM_POLY_COEFFS['C00']:.2f} м")
print(f"  Fallback офсет= {SPB_OFFSET_FALLBACK:.2f} м")

In [ ]:
# ======================================================================================
# МОДУЛЬ 0: ICESat-2 ATL08 — загрузка и 3D-корегистрация по Nuth & Kääb (2011)
# ======================================================================================
#
# ТЕОРИЯ (из исследования):
#   Сырые полосы ArcticDEM имеют абсолютное смещение до 4 м из-за отсутствия GCP.
#   ICESat-2 ATL08: вертикальная точность < 0.3 м (RMS), горизонтальная < 0.5 м.
#   Оба датума — WGS84, поэтому корегистрацию делаем ДО перехода к EGM2008.
#
#   Алгоритм Nuth & Kääb: ищем вектор (dx, dy, dz) минимизирующий RMSE(Δh),
#   используя зависимость Δh от уклона и экспозиции:
#     Δh = a·cos(b - ψ)·tan(α) + dz
#   где α — уклон, ψ — экспозиция, a = горизонтальный сдвиг, b — его направление.
# ======================================================================================

import os, gc, warnings, json
import numpy as np
import geopandas as gpd
import pandas as pd
import rioxarray
import rasterio
from rasterio.features import shapes, rasterize as rio_rasterize
from shapely.geometry import box, Point, mapping
from scipy import ndimage
from scipy.optimize import curve_fit
from scipy.interpolate import griddata
from osgeo import gdal
import folium

warnings.filterwarnings("ignore")
gdal.UseExceptions()


# ──────────────────────────────────────────────────────────────────────────────
# 0.1 Загрузка ATL08 треков через earthaccess
# ──────────────────────────────────────────────────────────────────────────────
def download_atl08_for_region(lat_min=59.5, lat_max=60.4,
                               lon_min=28.5, lon_max=31.5,
                               date_start="2018-10-14",
                               date_end="2023-12-31",
                               out_dir=None):
    """
    Загружает треки ICESat-2 ATL08 для области СПб через earthaccess API.

    Рекомендации по фильтрации (из исследования):
    - Приоритет: night_flag=1 (нет солнечного фона)
    - Исключить: cloud_flag_atm > 0, msw_flag > 0
    - Slope < 5° (иначе горизонтальная ошибка → вертикальная)
    - segment_terrain_height_uncertainty < 0.3 м

    Возвращает список путей к HDF5-файлам.
    """
    try:
        import earthaccess
        import h5py
    except ImportError:
        print("   ⚠ earthaccess/h5py не установлены: !pip install earthaccess h5py")
        return []

    out_dir = out_dir or os.path.join(WD, "atl08")
    os.makedirs(out_dir, exist_ok=True)

    # Проверка: уже скачано?
    existing = [f for f in os.listdir(out_dir) if f.endswith(".h5")]
    if existing:
        print(f"   ATL08: найдено {len(existing)} файлов в кэше ({out_dir})")
        return [os.path.join(out_dir, f) for f in existing]

    if not EARTHDATA_USER or not EARTHDATA_PASS:
        print("   ⚠ Нет NASA Earthdata логина (EARTHDATA_USER/EARTHDATA_PASS).")
        print("   → Регистрация: https://urs.earthdata.nasa.gov/")
        print("   → Используем synthetic-тест (заглушку)")
        return []

    print(f"   Поиск ATL08 для bbox [{lat_min},{lon_min}]–[{lat_max},{lon_max}]...")
    earthaccess.login(strategy="environment")

    results = earthaccess.search_data(
        short_name="ATL08",
        bounding_box=(lon_min, lat_min, lon_max, lat_max),
        temporal=(date_start, date_end),
        count=50
    )
    print(f"   Найдено гранул: {len(results)}")

    if not results:
        return []

    files = earthaccess.download(results, local_path=out_dir)
    print(f"   Скачано: {len(files)} файлов")
    return files


# ──────────────────────────────────────────────────────────────────────────────
# 0.2 Парсинг и фильтрация ATL08 HDF5 файлов
# ──────────────────────────────────────────────────────────────────────────────
def parse_atl08_file(h5_path, crs_out,
                     lat_min=59.5, lat_max=60.4,
                     lon_min=28.5, lon_max=31.5,
                     night_only=True,
                     max_slope_deg=5.0,
                     max_uncertainty_m=0.3):
    """
    Извлекает точки рельефа из одного HDF5-файла ATL08.

    Фильтры (из исследования):
      - Только ночные треки (night_flag=1) — меньше солнечного фона
      - cloud_flag_atm == 0 (нет облачности)
      - msw_flag == 0 (нет многократного рассеяния)
      - segment_terrain_slope <= max_slope_deg
      - segment_terrain_height_uncertainty <= max_uncertainty_m
      - Географический bbox СПб

    Возвращает GeoDataFrame с колонками: x, y, h_te (WGS84 ellipsoidal), slope
    """
    try:
        import h5py
    except ImportError:
        return gpd.GeoDataFrame()

    records = []
    # ICESat-2 имеет 6 лучей: gt1l, gt1r, gt2l, gt2r, gt3l, gt3r
    beams = ["gt1l", "gt1r", "gt2l", "gt2r", "gt3l", "gt3r"]

    try:
        with h5py.File(h5_path, 'r') as f:
            for beam in beams:
                if beam not in f:
                    continue
                grp = f[beam]
                if 'land_segments' not in grp:
                    continue
                ls = grp['land_segments']

                # Основные переменные
                try:
                    lat   = ls['latitude'][:]            # °N WGS84
                    lon   = ls['longitude'][:]           # °E WGS84
                    # h_te — высота рельефа (terrain), эллипсоидальная WGS84
                    h_te  = ls['terrain/h_te_best_fit'][:]
                    slope = ls['terrain/terrain_slope'][:]
                    unc   = ls['terrain/h_te_uncertainty'][:]
                    cloud = ls['cloud_flag_atm'][:]
                    msw   = ls['msw_flag'][:]
                    night = ls['night_flag'][:]
                except KeyError as e:
                    # Некоторые поля могут иметь другие пути в разных версиях ATL08
                    print(f"     {beam}: отсутствует поле {e}, пропуск")
                    continue

                # Географический фильтр
                geo_mask = ((lat >= lat_min) & (lat <= lat_max) &
                            (lon >= lon_min) & (lon <= lon_max))

                # Качественные фильтры
                qual_mask = (cloud == 0) & (msw == 0)
                if night_only:
                    qual_mask &= (night == 1)

                slope_mask = (np.abs(slope) <= max_slope_deg)
                unc_mask   = (unc <= max_uncertainty_m) & (unc >= 0)

                # Физически допустимые высоты для СПб (WGS84): от -5 до 300 м
                height_mask = (h_te > -5) & (h_te < 300)

                final = geo_mask & qual_mask & slope_mask & unc_mask & height_mask

                if not np.any(final):
                    continue

                for i in np.where(final)[0]:
                    records.append({
                        'lat': float(lat[i]),
                        'lon': float(lon[i]),
                        'h_te': float(h_te[i]),
                        'slope': float(slope[i]),
                        'unc': float(unc[i]),
                        'night': int(night[i]),
                        'beam': beam
                    })

    except Exception as e:
        print(f"   ⚠ Ошибка чтения {os.path.basename(h5_path)}: {e}")
        return gpd.GeoDataFrame()

    if not records:
        return gpd.GeoDataFrame()

    df = pd.DataFrame(records)
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df['lon'], df['lat']),
        crs="EPSG:4326"
    ).to_crs(crs_out)

    # Добавляем метрические координаты
    gdf['x'] = gdf.geometry.x
    gdf['y'] = gdf.geometry.y

    return gdf


def load_all_atl08(h5_files, crs_out, **kwargs):
    """Загружает все ATL08-файлы и объединяет в один GeoDataFrame."""
    parts = []
    for f in h5_files:
        gdf = parse_atl08_file(f, crs_out, **kwargs)
        if not gdf.empty:
            parts.append(gdf)
            print(f"   {os.path.basename(f)}: {len(gdf)} точек")

    if not parts:
        print("   ATL08: нет валидных точек")
        return gpd.GeoDataFrame()

    result = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=crs_out)
    print(f"   ATL08 итого: {len(result):,} точек рельефа")
    return result


# ──────────────────────────────────────────────────────────────────────────────
# 0.3 Маска «стабильных открытых» поверхностей (не лес, не застройка, не вода)
# ──────────────────────────────────────────────────────────────────────────────
def create_stable_surface_mask(dsm_path, crs, res=10, out_path=None):
    """
    Создаёт растровую маску «открытого рельефа» (bare earth) для корегистрации.

    Корегистрацию НЕЛЬЗЯ делать на:
      - Зданиях (DSM даёт крыши)
      - Лесе (DSM даёт полог)
      - Воде (SETSM генерирует артефакты)

    Стратегия: загружаем из OSM полигоны зданий, воды, леса → инвертируем.
    """
    import osmnx as ox
    out_path = out_path or os.path.join(WD, "stable_mask.tif")

    if os.path.exists(out_path):
        print("   stable_mask: из кэша")
        return out_path

    # Открываем DEM для получения метаданных
    dsm_10m = os.path.join(WD, "dsm_10m_stable.tif")
    if not os.path.exists(dsm_10m):
        opts = gdal.WarpOptions(format="GTiff", xRes=res, yRes=res,
                                resampleAlg="bilinear", dstNodata=-9999,
                                creationOptions=["COMPRESS=LZW"])
        gdal.Warp(dsm_10m, dsm_path, options=opts)

    with rasterio.open(dsm_10m) as src:
        mask = np.ones(src.shape, dtype=np.uint8)
        nd = src.nodata
        arr = src.read(1)
        if nd is not None:
            mask[arr == nd] = 0
        mask[np.isnan(arr)] = 0
        transform = src.transform
        profile = src.profile.copy()
        bounds = src.bounds

    # Bbox в WGS84 для OSM-запросов
    bbox_gdf = gpd.GeoDataFrame(
        {'geometry': [box(bounds.left, bounds.bottom, bounds.right, bounds.top)]},
        crs=crs
    )
    b = bbox_gdf.to_crs("EPSG:4326").total_bounds
    bbox_osm = (b[0], b[1], b[2], b[3])  # (west, south, east, north)

    ox.settings.timeout = 120

    # Категории поверхностей для ИСКЛЮЧЕНИЯ из корегистрации
    exclusion_tags = [
        ({'building': True},          "здания"),
        ({'natural': ['water', 'wetland']}, "вода/болота"),
        ({'landuse': ['forest', 'wood', 'residential', 'industrial']}, "лес/застройка"),
        ({'natural': 'wood'},          "лес OSM"),
    ]

    for tags, label in exclusion_tags:
        try:
            feats = ox.features_from_bbox(bbox=bbox_osm, tags=tags)
            if feats.empty:
                continue
            feats = feats[feats.geometry.type.isin(['Polygon', 'MultiPolygon'])]
            feats = feats.to_crs(crs)
            if feats.empty:
                continue
            excl_mask = rio_rasterize(
                [(g, 1) for g in feats.geometry],
                out_shape=mask.shape,
                transform=transform,
                fill=0, dtype='uint8'
            )
            n_excl = int(np.sum(excl_mask))
            mask[excl_mask == 1] = 0
            print(f"   Исключено ({label}): {n_excl:,} пикселей")
        except Exception as e:
            print(f"   ⚠ OSM {label}: {e}")

    pct = 100 * np.sum(mask) / mask.size
    print(f"   Стабильных пикселей: {np.sum(mask):,} ({pct:.1f}%)")

    profile.update(dtype='uint8', count=1, nodata=255)
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(mask, 1)

    del arr, mask; gc.collect()
    return out_path


# ──────────────────────────────────────────────────────────────────────────────
# 0.4 Алгоритм Nuth & Kääb (2011) — 3D корегистрация
# ──────────────────────────────────────────────────────────────────────────────
def nuth_kaab_coregistration(dsm_path, atl08_gdf, stable_mask_path,
                               crs, max_iter=10, conv_threshold=0.05):
    """
    Реализация алгоритма Nuth & Kääb (2011) для корегистрации ArcticDEM по ICESat-2.

    Теория:
      Δh = a·cos(b - ψ)·tan(α) + dz
      где a = |horizontal_shift|, b = direction, ψ = aspect, α = slope
      → итеративно находим dx, dy (из a,b) и dz (медиана остатков)

    Аргументы:
      dsm_path       — путь к GeoTIFF DSM (WGS84 эллипсоидальные высоты)
      atl08_gdf      — GeoDataFrame с колонками x, y, h_te (в crs)
      stable_mask_path — растр маски "открытого рельефа" (1=ok, 0=исключить)
      max_iter       — максимум итераций
      conv_threshold — порог сходимости (м)

    Возвращает:
      dict: dx, dy, dz (в метрах), rmse_before, rmse_after, n_points
    """
    if atl08_gdf.empty:
        print("   Nuth&Kääb: нет точек ATL08, возвращаем нулевой сдвиг")
        return {'dx': 0.0, 'dy': 0.0, 'dz': 0.0,
                'rmse_before': np.nan, 'rmse_after': np.nan, 'n_points': 0}

    print(f"   Nuth&Kääb: {len(atl08_gdf):,} ATL08-точек, max_iter={max_iter}")

    # Загружаем DEM на пониженном разрешении для скорости
    dsm_4m = os.path.join(WD, "dsm_4m_nk.tif")
    if not os.path.exists(dsm_4m):
        opts = gdal.WarpOptions(format="GTiff", xRes=4, yRes=4,
                                resampleAlg="bilinear", dstNodata=-9999,
                                creationOptions=["COMPRESS=LZW"])
        gdal.Warp(dsm_4m, dsm_path, options=opts)

    with rasterio.open(dsm_4m) as src:
        dsm_arr  = src.read(1).astype(np.float32)
        nd = src.nodata
        if nd is not None:
            dsm_arr[dsm_arr == nd] = np.nan
        transform = src.transform
        res_x = transform.a
        res_y = abs(transform.e)

    # Вычисляем уклон и экспозицию из DSM
    def compute_slope_aspect(dem, dx, dy):
        """Уклон (рад) и экспозиция (рад) из массива высот."""
        dz_dx = ndimage.sobel(np.nan_to_num(dem), axis=1) / (8 * dx)
        dz_dy = ndimage.sobel(np.nan_to_num(dem), axis=0) / (8 * dy)
        slope  = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
        aspect = np.arctan2(-dz_dy, dz_dx)
        return slope, aspect

    slope_arr, aspect_arr = compute_slope_aspect(dsm_arr, res_x, res_y)

    # Загружаем маску
    if stable_mask_path and os.path.exists(stable_mask_path):
        with rasterio.open(stable_mask_path) as ms:
            smask = ms.read(1)
        if smask.shape != dsm_arr.shape:
            # Ресамплируем маску
            from scipy.ndimage import zoom
            zy = dsm_arr.shape[0] / smask.shape[0]
            zx = dsm_arr.shape[1] / smask.shape[1]
            smask = zoom(smask, (zy, zx), order=0)
            smask = smask[:dsm_arr.shape[0], :dsm_arr.shape[1]]
    else:
        smask = np.ones_like(dsm_arr, dtype=np.uint8)

    # Кумулятивный сдвиг
    cum_dx, cum_dy, cum_dz = 0.0, 0.0, 0.0
    rmse_before = None

    def sample_dsm_at_points(pts_x, pts_y, dem, transform):
        """Билинейная интерполяция DEM в точках (x, y)."""
        c = (pts_x - transform.c) / transform.a
        r = (pts_y - transform.f) / transform.e

        ci = np.clip(c.astype(int), 0, dem.shape[1] - 2)
        ri = np.clip(r.astype(int), 0, dem.shape[0] - 2)

        # Билинейные веса
        fc = c - ci
        fr = r - ri

        v00 = dem[ri,     ci]
        v01 = dem[ri,     ci+1]
        v10 = dem[ri+1,   ci]
        v11 = dem[ri+1,   ci+1]

        vals = ((1-fr)*(1-fc)*v00 + (1-fr)*fc*v01 +
                    fr*(1-fc)*v10 +     fr*fc*v11)
        # Исключаем NaN
        nan_mask = (np.isnan(v00) | np.isnan(v01) |
                    np.isnan(v10) | np.isnan(v11))
        vals[nan_mask] = np.nan
        return vals

    def sample_raster_at_points(pts_x, pts_y, arr, transform):
        """Nearest-neighbor сэмплирование массива."""
        c = np.clip(((pts_x - transform.c) / transform.a).astype(int),
                     0, arr.shape[1]-1)
        r = np.clip(((pts_y - transform.f) / transform.e).astype(int),
                     0, arr.shape[0]-1)
        return arr[r, c]

    pts_x = atl08_gdf['x'].values
    pts_y = atl08_gdf['y'].values
    pts_h = atl08_gdf['h_te'].values

    for iteration in range(max_iter):
        # Смещаем координаты точек на кумулятивный сдвиг
        px = pts_x - cum_dx
        py = pts_y - cum_dy

        # Сэмплируем DSM, slope, aspect в точках ATL08
        dsm_at_pts  = sample_dsm_at_points(px, py, dsm_arr, transform)
        slope_at_pts = sample_raster_at_points(px, py, slope_arr, transform)
        aspect_at_pts = sample_raster_at_points(px, py, aspect_arr, transform)
        stable_at_pts = sample_raster_at_points(px, py, smask, transform)

        # Δh = DSM - ATL08 (положительный → DSM выше ATL08)
        dh = dsm_at_pts - (pts_h + cum_dz)

        # Фильтр валидных точек
        valid = (
            ~np.isnan(dsm_at_pts) &
            ~np.isnan(dh) &
            (stable_at_pts == 1) &
            (slope_at_pts > np.deg2rad(1.0)) &   # нужен уклон для нормализации
            (slope_at_pts < np.deg2rad(30.0)) &   # слишком крутые → ошибки
            (np.abs(dh) < 50)                      # грубые выбросы
        )

        n_valid = int(np.sum(valid))
        if n_valid < 20:
            print(f"   Итерация {iteration+1}: недостаточно точек ({n_valid}), стоп")
            break

        dh_v  = dh[valid]
        sl_v  = slope_at_pts[valid]
        asp_v = aspect_at_pts[valid]

        rmse = float(np.sqrt(np.mean(dh_v**2)))
        if rmse_before is None:
            rmse_before = rmse

        # Нормализованная разность высот: Δh / tan(slope)
        dh_norm = dh_v / np.tan(sl_v)

        # Подгонка: dh_norm = a·cos(b - aspect) + c
        # → параметры: a=амплитуда, b=фаза, c=вертикальный сдвиг
        def model_nk(aspect, a, b, c):
            return a * np.cos(b - aspect) + c

        try:
            # p0: начальные приближения (сдвиг ~1 м, случайное направление)
            p0 = [1.0, 0.0, float(np.median(dh_norm))]
            popt, _ = curve_fit(model_nk, asp_v, dh_norm, p0=p0,
                                maxfev=2000, method='lm')
            a_fit, b_fit, c_fit = popt
        except RuntimeError:
            print(f"   Итерация {iteration+1}: curve_fit не сошёлся")
            break

        # Горизонтальный сдвиг в метрах
        # Знак и направление: dx = a·sin(b), dy = a·cos(b)
        # (соответствует сдвигу в проекции UTM)
        shift_mag = float(a_fit)     # метры
        shift_dir = float(b_fit)     # радианы
        iter_dx = shift_mag * np.sin(shift_dir)
        iter_dy = shift_mag * np.cos(shift_dir)

        # Вертикальный сдвиг = медиана остатков после горизонтальной коррекции
        dh_residual = dh_v - model_nk(asp_v, a_fit, b_fit, 0.0)
        iter_dz = float(np.median(dh_residual))

        cum_dx += iter_dx
        cum_dy += iter_dy
        cum_dz += iter_dz

        shift_total = np.sqrt(iter_dx**2 + iter_dy**2 + iter_dz**2)
        print(f"   Итерация {iteration+1:2d}: RMSE={rmse:.3f}м, "
              f"Δ=({iter_dx:+.3f}, {iter_dy:+.3f}, {iter_dz:+.3f})м, "
              f"N={n_valid:,}")

        if shift_total < conv_threshold:
            print(f"   Сходимость достигнута (|Δ|={shift_total:.4f} м < {conv_threshold} м)")
            break

    # Финальный RMSE
    px_final = pts_x - cum_dx
    py_final = pts_y - cum_dy
    dsm_final = sample_dsm_at_points(px_final, py_final, dsm_arr, transform)
    dh_final  = dsm_final - (pts_h + cum_dz)
    valid_f   = ~np.isnan(dh_final) & (np.abs(dh_final) < 50)
    rmse_after = float(np.sqrt(np.mean(dh_final[valid_f]**2))) if np.any(valid_f) else np.nan

    result = {
        'dx': float(cum_dx),
        'dy': float(cum_dy),
        'dz': float(cum_dz),
        'rmse_before': float(rmse_before) if rmse_before else np.nan,
        'rmse_after':  rmse_after,
        'n_points':    n_valid
    }

    print(f"\n   ══ Nuth & Kääb результат ══")
    print(f"   Вектор сдвига: dx={cum_dx:+.3f} м, dy={cum_dy:+.3f} м, dz={cum_dz:+.3f} м")
    print(f"   Горизонт. сдвиг: {np.sqrt(cum_dx**2+cum_dy**2):.3f} м")
    print(f"   RMSE: {rmse_before:.3f} м → {rmse_after:.3f} м")
    if rmse_after < 0.5:
        print("   ✓ Абсолютная точность: < 0.5 м (соответствует требованиям)")
    elif rmse_after < 1.0:
        print("   ≈ Абсолютная точность: 0.5–1.0 м (приемлемо для DSM)")
    else:
        print("   ⚠ RMSE > 1.0 м — проверьте качество ATL08 данных")

    del dsm_arr, slope_arr, aspect_arr; gc.collect()
    return result


def apply_3d_shift_to_dsm(dsm_path, shift_dict, out_path, crs):
    """
    Применяет 3D-вектор сдвига (dx, dy, dz) к растру DSM.

    dx, dy — горизонтальный сдвиг (метры в UTM)
    dz     — вертикальный сдвиг (прибавляется к высотам)

    Примечание: горизонтальный сдвиг реализован через пересдвиг GeoTransform
    (сдвигаем координатную привязку, а не ресамплируем пиксели).
    Для субпиксельных сдвигов используется warp с новым origin.
    """
    dx = shift_dict.get('dx', 0.0)
    dy = shift_dict.get('dy', 0.0)
    dz = shift_dict.get('dz', 0.0)

    if abs(dx) < 0.01 and abs(dy) < 0.01 and abs(dz) < 0.01:
        print("   Сдвиг < 0.01 м — копируем без изменений")
        import shutil
        shutil.copy2(dsm_path, out_path)
        return out_path

    src_ds = gdal.Open(dsm_path)
    gt = list(src_ds.GetGeoTransform())

    # Новый origin (сдвиг координатной сетки)
    new_gt = gt.copy()
    new_gt[0] = gt[0] - dx   # X_min сдвигаем
    new_gt[3] = gt[3] - dy   # Y_max сдвигаем (dy — в UTM, Y возрастает вверх)

    drv = gdal.GetDriverByName('GTiff')
    dst = drv.CreateCopy(out_path, src_ds,
                          options=['COMPRESS=LZW', 'TILED=YES'])
    dst.SetGeoTransform(new_gt)

    # Применяем dz (вертикальный сдвиг)
    if abs(dz) > 0.001:
        band = dst.GetRasterBand(1)
        nd_val = band.GetNoDataValue()
        cols, rows = dst.RasterXSize, dst.RasterYSize
        tile_sz = 4096
        for row_off in range(0, rows, tile_sz):
            row_sz = min(tile_sz, rows - row_off)
            for col_off in range(0, cols, tile_sz):
                col_sz = min(tile_sz, cols - col_off)
                tile = band.ReadAsArray(col_off, row_off, col_sz, row_sz).astype(np.float32)
                valid = ~np.isnan(tile)
                if nd_val is not None:
                    valid &= (tile != nd_val)
                tile[valid] += dz
                band.WriteArray(tile, col_off, row_off)

    dst.FlushCache()
    dst = None
    src_ds = None
    print(f"   Скорректированный DSM: {out_path}")
    return out_path


print("[MODULE 0] ICESat-2 ATL08 + Nuth & Kääb — функции загружены")

In [ ]:
# ======================================================================================
# МОДУЛЬ 1: ВЕРТИКАЛЬНАЯ КАЛИБРОВКА (улучшенная)
# ======================================================================================

# ──────────────────────────────────────────────────────────────────────────────
# 1.1 EGM2008 — 2D-полиномиальная аппроксимация для СПб
# ──────────────────────────────────────────────────────────────────────────────
def egm2008_undulation(lat, lon):
    """
    Ундуляция геоида EGM2008 (N) в точке (lat, lon).
    N = h_ellipsoidal - H_orthometric_EGM2008

    Иерархия методов:
    1. pyproj с сеточным файлом EGM2008 (наиболее точный)
    2. 2D-полиномиальная поверхность степени 2 (калибрована по GPS-нивелированию СПб)
    3. Константа fallback 16.1 м (если выход за область применимости)

    ВАЖНО: Это N для EGM2008, а НЕ для BS-77.
    Переход к BS-77 требует дополнительной поправки DELTA_BS77 = +0.49 м.
    """
    # Метод 1: pyproj с сеточным файлом (Kaggle/Colab часто не имеет сетки)
    try:
        from pyproj import Transformer
        t = Transformer.from_crs("EPSG:4979", "EPSG:4326+3855", always_xy=True)
        _, _, h_egm = t.transform(lon, lat, 0.0)
        N = -h_egm
        # КРИТИЧЕСКИ ВАЖНО: если pyproj вернул ~0 — сетки нет
        if abs(N) < 0.5:
            raise ValueError(f"EGM grid unavailable (N={N:.4f} м → fallback)")
        return float(N)
    except Exception:
        pass

    # Метод 2: 2D-полиномиальная поверхность степени 2
    # Кросс-проверка с EGM2008 online calc: ошибка < 0.05 м в диапазоне 59.5–60.4°N, 28.5–31.5°E
    c = EGM_POLY_COEFFS
    dlat = lat - EGM_POLY_LAT0
    dlon = lon - EGM_POLY_LON0
    N_approx = (c['C00']
                + c['C10'] * dlat
                + c['C01'] * dlon
                + c['C20'] * dlat**2
                + c['C11'] * dlat * dlon
                + c['C02'] * dlon**2)

    # Проверка: значение должно быть в физически разумном диапазоне для СПб
    if 14.0 <= N_approx <= 18.5:
        return float(N_approx)

    # Метод 3: константа
    return float(EGM_POLY_COEFFS['C00'])


def egm2008_to_bs77(h_egm2008, lat=59.93, lon=30.32):
    """
    Переводит высоты из датума EGM2008 в Балтийскую систему высот 1977 (BS-77).

    Формула: H_BS77 = H_EGM2008 - delta_BS77

    Физический смысл DELTA_BS77 = +0.49 м:
      Нуль Кронштадтского футштока (BS-77) расположен ВЫШЕ поверхности
      глобального геоида EGM2008 на 0.49 ± 0.06 м для региона СПб.
      Это означает: Балтийское море (BS-77 = 0 м) находится НИЖЕ
      «глобального уровня моря» EGM2008 примерно на 0.49 м.

    Источник: GPS-нивелирование GEOBRIDGE, 12 точек в ЛО,
              σ = 0.06 м, диапазон [+0.34, +0.69] м.
    """
    return h_egm2008 - DELTA_BS77


# ──────────────────────────────────────────────────────────────────────────────
# 1.2 Вспомогательные функции чтения
# ──────────────────────────────────────────────────────────────────────────────
def read_dem_at_point(dem_path, lat, lon, crs, radius_px=10):
    """Медиана DEM в окне radius_px вокруг точки (без загрузки всего файла)."""
    pt = gpd.GeoDataFrame({'geometry': [Point(lon, lat)]}, crs="EPSG:4326").to_crs(crs)
    px, py = pt.geometry.iloc[0].x, pt.geometry.iloc[0].y

    ds = gdal.Open(dem_path)
    gt = ds.GetGeoTransform()
    c  = int((px - gt[0]) / gt[1])
    r  = int((py - gt[3]) / gt[5])

    x0 = max(0, c - radius_px);  y0 = max(0, r - radius_px)
    xs = min(2 * radius_px, ds.RasterXSize - x0)
    ys = min(2 * radius_px, ds.RasterYSize - y0)

    if xs <= 0 or ys <= 0:
        ds = None; return np.nan

    arr = ds.GetRasterBand(1).ReadAsArray(x0, y0, xs, ys).astype(float)
    nd  = ds.GetRasterBand(1).GetNoDataValue()
    ds  = None

    if nd is not None:
        arr[arr == nd] = np.nan
    valid = arr[~np.isnan(arr)]
    valid = valid[(valid > -5) & (valid < 300)]  # WGS84 диапазон для СПб
    return float(np.median(valid)) if len(valid) > 0 else np.nan


def get_water_levels(csv_path, date_str, hour_start=15, hour_end=16):
    """Уровни с постов в момент съёмки (см → м, округлено до 0.01)."""
    levels = {}
    try:
        df = pd.read_csv(csv_path)
        df['_dt'] = pd.to_datetime(df.iloc[:, 0])
        day = df[df['_dt'].dt.strftime('%Y-%m-%d') == date_str]
        win = day[(day['_dt'].dt.hour >= hour_start) & (day['_dt'].dt.hour <= hour_end)]
        for col in ['ГИ', 'С1 ФЗ', 'С2 ФЗ']:
            if col in win.columns:
                v = pd.to_numeric(win[col], errors='coerce').dropna() / 100
                if not v.empty:
                    levels[col] = round(float(v.mean()), 2)
    except Exception as e:
        print(f"   ⚠ CSV: {e}")
    return levels


# ──────────────────────────────────────────────────────────────────────────────
# 1.3 Сезонная поправка (leaf-on / leaf-off)
# ──────────────────────────────────────────────────────────────────────────────
def get_leaf_status(date_str):
    """
    Определяет фенологическое состояние растительности по дате съёмки.

    Для СПб (60°N):
      - Leaf-OFF: октябрь–апрель (месяцы 10–4) → оптика проникает глубже
      - Leaf-ON:  май–сентябрь (месяцы 5–9) → DSM выше истинного рельефа

    Влияние на DTM-фильтрацию:
      Leaf-ON  → увеличиваем жёсткость CSF (cloth_rigidness) на 30%
      Leaf-OFF → стандартные параметры CSF

    Источник: исследование 2025, раздел «Влияние сезонности».
    """
    try:
        from datetime import datetime
        dt = datetime.strptime(date_str, "%Y-%m-%d")
        month = dt.month
        if 5 <= month <= 9:
            return "leaf_on", 1.3   # множитель жёсткости ткани CSF
        else:
            return "leaf_off", 1.0
    except Exception:
        return "unknown", 1.0


# ──────────────────────────────────────────────────────────────────────────────
# 1.4 Основная функция калибровки (с поправкой BS-77 и бутстрэпом)
# ──────────────────────────────────────────────────────────────────────────────
def calibrate(dem_path, control_pts, csv_path, crs, n_bootstrap=500):
    """
    Вычисляет total_offset и его неопределённость.

    Формула:
      H_BS77 = DSM_WGS84 - offset
      offset = N_EGM2008 + delta_BS77
             ≈ 16.1 + 0.49 ≈ 16.59 м  (теоретически)

    НО: водомерные посты дают эмпирический offset ≈ 16.09 м,
    что уже учитывает оба члена + локальные аномалии ArcticDEM.

    Иерархия источников:
      1. Первичный водомерный пост ГИ (наименьшие артефакты DSM)
      2. Медиана водных постов
      3. Медиана наземных точек (ВПП)
      4. Константа SPB_OFFSET_FALLBACK

    Бутстрэп-оценка 95% CI: повторная выборка из всех валидных точек.
    """
    print("   Контрольные точки:")
    header = f"{'Точка':<28s} {'DSM':>7s} {'N_EGM':>7s} {'H_EGM':>7s} {'H_BS77':>7s} {'Offset':>8s} {'Тип'}"
    print(f"   {header}")
    print(f"   {'-'*85}")

    wl = get_water_levels(csv_path, STRIP_DATE)
    if wl:
        print(f"   Уровни {STRIP_DATE}: " + ", ".join(f"{k}={v:.2f}" for k, v in wl.items()))
    else:
        print(f"   ⚠ CSV не загружен — водомерные посты будут пропущены")

    # Сезонный статус
    leaf_status, csf_mult = get_leaf_status(STRIP_DATE)
    print(f"   Фенология {STRIP_DATE}: {leaf_status} (CSF-множитель: ×{csf_mult:.1f})")

    water_offsets  = []
    ground_offsets = []
    primary_offset = None
    first_water    = True
    all_offsets_meta = []

    col_map = {"ГИ (на воде)": "ГИ", "С1 ФЗ (на воде)": "С1 ФЗ", "С2 ФЗ (на воде)": "С2 ФЗ"}

    for name, lat, lon, h_known, ptype in control_pts:
        dsm_val = read_dem_at_point(dem_path, lat, lon, crs)
        n_egm   = egm2008_undulation(lat, lon)

        if np.isnan(dsm_val):
            print(f"   {name:<28s} N/A")
            continue

        # Ортометрическая высота в системе EGM2008
        h_egm = dsm_val - n_egm
        # Высота в BS-77 (теоретически)
        h_bs77_theoretical = egm2008_to_bs77(h_egm, lat, lon)

        if ptype == "water":
            csv_col = col_map.get(name)
            h_bs77 = wl.get(csv_col) if csv_col else None
            if h_bs77 is None:
                print(f"   {name:<28s} нет CSV — пропуск")
                continue
        else:
            h_bs77 = h_known

        # Эмпирический total offset
        offset_i = dsm_val - h_bs77
        in_range = SPB_OFFSET_MIN <= offset_i <= SPB_OFFSET_MAX
        status   = "✓" if in_range else "⚠"

        print(f"   {name:<28s} {dsm_val:>7.2f} {n_egm:>7.2f} {h_egm:>7.2f} "
              f"{h_bs77:>7.2f} {offset_i:>+8.2f} {'вода' if ptype=='water' else 'земля'} {status}")

        if not in_range:
            print(f"   {'':28s} ⚠ ИСКЛЮЧЕНО (вне [{SPB_OFFSET_MIN}, {SPB_OFFSET_MAX}] м)")
            continue

        all_offsets_meta.append({'offset': offset_i, 'type': ptype,
                                  'name': name, 'lat': lat, 'lon': lon})
        if ptype == "water":
            water_offsets.append(offset_i)
            if first_water:
                primary_offset = offset_i
                first_water = False
        else:
            ground_offsets.append(offset_i)

    # Выбор итогового offset
    if primary_offset is not None:
        offset = primary_offset
        note   = "первичный пост ГИ"
    elif water_offsets:
        offset = float(np.median(water_offsets))
        note   = f"медиана водных постов (N={len(water_offsets)})"
    elif ground_offsets:
        offset = float(np.median(ground_offsets))
        note   = f"медиана ВПП (N={len(ground_offsets)})"
    else:
        offset = SPB_OFFSET_FALLBACK
        note   = "константа fallback (нет данных)"

    # Бутстрэп 95% CI
    ci_low, ci_high = offset, offset
    all_offsets_vals = [m['offset'] for m in all_offsets_meta]
    if len(all_offsets_vals) >= 3:
        rng = np.random.default_rng(42)
        boot_medians = []
        for _ in range(n_bootstrap):
            sample = rng.choice(all_offsets_vals, size=len(all_offsets_vals), replace=True)
            boot_medians.append(np.median(sample))
        ci_low  = float(np.percentile(boot_medians, 2.5))
        ci_high = float(np.percentile(boot_medians, 97.5))
        print(f"\n   Bootstrap 95% CI: [{ci_low:.3f}, {ci_high:.3f}] м (N={n_bootstrap})")

    # Разброс между постами — диагностика разнородности стрипов
    if len(water_offsets) > 1:
        spread = max(water_offsets) - min(water_offsets)
        warn = " ⚠ ВНИМАНИЕ" if spread > 0.5 else ""
        print(f"   Разброс водных постов: {spread:.3f} м  (допустимо ≤ 0.5 м){warn}")
        if spread > 0.5:
            print("   → Разные ArcticDEM-стрипы имеют разные смещения!")
            print("   → Рекомендуется пространственно-переменный offset (см. Module 1.5)")

    print(f"\n   {'─'*60}")
    print(f"   Принятый offset: {offset:+.4f} м  [{note}]")
    print(f"   Порог затопления: {FLOOD_LEVEL} + {offset:.4f} = {FLOOD_LEVEL + offset:.4f} м (DSM)")
    print(f"   Теоретический (N_EGM + δ_BS77): {EGM_POLY_COEFFS['C00']:.2f} + {DELTA_BS77:.2f} = "
          f"{EGM_POLY_COEFFS['C00'] + DELTA_BS77:.2f} м")
    delta_theory = offset - (EGM_POLY_COEFFS['C00'] + DELTA_BS77)
    print(f"   Δ(эмпирический - теоретический): {delta_theory:+.3f} м "
          f"({'в норме' if abs(delta_theory) < 0.5 else '⚠ большое расхождение!'})")

    return {
        'offset':   offset,
        'ci_low':   ci_low,
        'ci_high':  ci_high,
        'note':     note,
        'csf_mult': csf_mult,
        'all_meta': all_offsets_meta
    }


# ──────────────────────────────────────────────────────────────────────────────
# 1.5 Пространственно-переменный offset (тайловая карта)
# ──────────────────────────────────────────────────────────────────────────────
def build_spatial_offset_map(dsm_path, calib_result, crs, tile_km=5):
    """
    Строит растровую карту offset на основе метаданных контрольных точек.

    Проблема (из исследования):
      Разные стрипы ArcticDEM (захватывающие разные части города)
      могут иметь разные абсолютные смещения.
      Использование единого offset даёт ошибки на границах стрипов.

    Решение:
      - На каждой контрольной точке вычислен локальный offset
      - Интерполируем IDW-методом на регулярную сетку
      - Ограничиваем диапазон [SPB_OFFSET_MIN, SPB_OFFSET_MAX]

    Если точек < 3 — возвращаем константный растр.
    """
    meta = calib_result.get('all_meta', [])
    offset_const = calib_result['offset']

    # Пересемплируем DSM до tile_km для построения карты offset
    tile_m = tile_km * 1000
    dsm_coarse = os.path.join(WD, f"dsm_{tile_km}km.tif")
    out_path = os.path.join(WD, "offset_map.tif")

    opts = gdal.WarpOptions(format="GTiff", xRes=tile_m, yRes=tile_m,
                            resampleAlg="bilinear", dstNodata=-9999,
                            creationOptions=["COMPRESS=LZW"])
    gdal.Warp(dsm_coarse, dsm_path, options=opts)

    with rasterio.open(dsm_coarse) as src:
        shape = src.shape
        transform = src.transform
        profile = src.profile.copy()

    if len(meta) < 3:
        # Константный растр
        arr = np.full(shape, offset_const, dtype=np.float32)
        print(f"   Offset-карта: константа {offset_const:.3f} м (мало точек)")
    else:
        # IDW-интерполяция
        ctrl_pts = []
        for m in meta:
            pt_gdf = gpd.GeoDataFrame(
                {'geometry': [Point(m['lon'], m['lat'])]}, crs="EPSG:4326"
            ).to_crs(crs)
            ctrl_pts.append((pt_gdf.geometry.iloc[0].x,
                             pt_gdf.geometry.iloc[0].y,
                             m['offset']))

        # Координаты центров пикселей карты offset
        rows_i = np.arange(shape[0])
        cols_i = np.arange(shape[1])
        xi = transform.c + (cols_i + 0.5) * transform.a
        yi = transform.f + (rows_i + 0.5) * transform.e
        XI, YI = np.meshgrid(xi, yi)

        pts_x = np.array([p[0] for p in ctrl_pts])
        pts_y = np.array([p[1] for p in ctrl_pts])
        pts_o = np.array([p[2] for p in ctrl_pts])

        arr = griddata((pts_x, pts_y), pts_o, (XI, YI), method='linear')

        # Заполняем NaN за пределами hull константой
        arr_nn = griddata((pts_x, pts_y), pts_o, (XI, YI), method='nearest')
        nan_mask = np.isnan(arr)
        arr[nan_mask] = arr_nn[nan_mask]

        arr = np.clip(arr, SPB_OFFSET_MIN, SPB_OFFSET_MAX).astype(np.float32)
        print(f"   Offset-карта: IDW-интерполяция, "
              f"диапазон [{arr.min():.2f}, {arr.max():.2f}] м")

    profile.update(dtype='float32', count=1, nodata=-9999)
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(arr, 1)

    return out_path


print("[MODULE 1] Вертикальная калибровка — функции загружены")

In [ ]:
# ======================================================================================
# МОДУЛЬ 2–3: МАСКА ПОКРЫТИЯ + DSM → DTM (улучшенный)
# ======================================================================================

def create_coverage_mask(dsm_path, out_path, crs, res=10):
    """Маска покрытия DEM (без изменений)."""
    print("   Создание маски покрытия из nodata...")
    dsm_10m = os.path.join(WD, "dsm_10m_cov.tif")
    if not os.path.exists(dsm_10m):
        opts = gdal.WarpOptions(format="GTiff", xRes=res, yRes=res,
                                resampleAlg="bilinear", dstNodata=-9999,
                                creationOptions=["COMPRESS=LZW", "TILED=YES"])
        gdal.Warp(dsm_10m, dsm_path, options=opts)

    with rasterio.open(dsm_10m) as src:
        arr  = src.read(1)
        nd   = src.nodata
        mask = np.ones_like(arr, dtype=np.uint8)
        if nd is not None:
            mask[arr == nd] = 0
        mask[np.isnan(arr)] = 0
        profile = src.profile.copy()

    struct = ndimage.generate_binary_structure(2, 2)
    mask_closed = ndimage.binary_closing(mask, structure=struct, iterations=5).astype(np.uint8)

    try:
        import osmnx as ox
        ox.settings.timeout = 60
        city = ox.geocode_to_gdf("Saint Petersburg, Russia").to_crs(crs)
        with rasterio.open(dsm_10m) as src2:
            cov_transform = src2.transform
        city_mask = rio_rasterize(
            [(g, 1) for g in city.geometry],
            out_shape=mask_closed.shape,
            transform=cov_transform, fill=0, dtype='uint8'
        )
        mask_closed = mask_closed & city_mask
        print("   + клиппинг по границе города")
    except Exception as e:
        print(f"   ⚠ Граница города: {e}")

    profile.update(dtype='uint8', count=1, nodata=255)
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(mask_closed, 1)

    pct = 100 * np.sum(mask_closed) / mask_closed.size
    print(f"   Валидных: {np.sum(mask_closed):,} пикселей ({pct:.1f}%)")
    del arr, mask, mask_closed; gc.collect()
    return out_path


# ──────────────────────────────────────────────────────────────────────────────
# 3.1 Water Flattening — выравнивание водных поверхностей
# ──────────────────────────────────────────────────────────────────────────────
def apply_water_flattening(dsm_path, out_path, crs, offset, gauge_wl=None):
    """
    Принудительное выравнивание водных поверхностей (Water Flattening).

    Проблема:
      Алгоритм SETSM генерирует шумовые артефакты (волны, спайки) на воде
      из-за отсутствия стабильных текстурных ориентиров для стереосопоставления.

    Решение:
      1. Акватория Финского залива и Ладожского озера: H_BS77 = 0.00 м
         (в DSM-координатах: DSM = offset + 0.00)
      2. Река Нева: монотонный градиент от +5.1 м (исток, Ладога)
         до 0.00 м (устье, Финский залив)
      3. Прочие реки и озёра: H_BS77 = среднему по контуру

    Источник масок: OSM (natural=water, waterway=river)
    """
    import osmnx as ox
    import shutil

    # Читаем метаданные DSM
    with rasterio.open(dsm_path) as src:
        profile = src.profile.copy()
        bounds  = src.bounds
        shape   = src.shape
        transform = src.transform

    bbox_gdf = gpd.GeoDataFrame(
        {'geometry': [box(bounds.left, bounds.bottom, bounds.right, bounds.top)]},
        crs=crs
    )
    b = bbox_gdf.to_crs("EPSG:4326").total_bounds
    bbox_osm = (b[0], b[1], b[2], b[3])

    ox.settings.timeout = 120

    # ── Загружаем водные полигоны из OSM ──
    water_polygons = []
    try:
        water_feats = ox.features_from_bbox(
            bbox=bbox_osm,
            tags={'natural': 'water', 'water': True}
        )
        if not water_feats.empty:
            water_feats = water_feats[
                water_feats.geometry.type.isin(['Polygon', 'MultiPolygon'])
            ].to_crs(crs)
            water_polygons = list(water_feats.geometry)
            print(f"   Water flattening: {len(water_polygons)} водных полигонов")
    except Exception as e:
        print(f"   ⚠ OSM water: {e}")

    if not water_polygons:
        print("   Water flattening: нет данных OSM — пропуск")
        shutil.copy2(dsm_path, out_path)
        return out_path

    # ── Растеризуем водные маски ──
    with rasterio.open(dsm_path) as src:
        arr = src.read(1).astype(np.float32)
        nd  = src.nodata
        if nd is not None:
            arr[arr == nd] = np.nan

    water_mask = rio_rasterize(
        [(g, 1) for g in water_polygons],
        out_shape=shape,
        transform=transform,
        fill=0, dtype='uint8'
    )

    # DSM-значение для уровня моря (H_BS77 = 0.00 м)
    sea_level_dsm = offset + 0.00

    # ── Применяем выравнивание ──
    n_flattened = int(np.sum(water_mask))
    arr[water_mask == 1] = sea_level_dsm

    # ── Нева: монотонный градиент (исток Ладога ~5.1 м → устье 0.0 м BS-77) ──
    try:
        neva_feats = ox.features_from_bbox(
            bbox=bbox_osm,
            tags={'waterway': 'river', 'name': 'Нева'}
        )
        if not neva_feats.empty and not neva_feats.geometry.isnull().all():
            neva_feats = neva_feats.to_crs(crs)
            # Упрощённый вариант: линейный градиент от восток (Ладога, ~5.1 м) к западу (залив, 0 м)
            # В DSM-координатах: DSM_Neva(x) = offset + 5.1 * (x_max - x) / (x_max - x_min)
            neva_total_bounds = neva_feats.total_bounds
            x_min_neva = neva_total_bounds[0]
            x_max_neva = neva_total_bounds[2]

            # Растеризуем буфер Невы отдельно
            neva_buf = neva_feats.geometry.buffer(200)  # 200 м буфер по обе стороны
            neva_mask = rio_rasterize(
                [(g, 1) for g in neva_buf],
                out_shape=shape, transform=transform,
                fill=0, dtype='uint8'
            )

            # Координаты X каждого пикселя
            col_coords = np.arange(shape[1]) * transform.a + transform.c
            X_grid = np.tile(col_coords, (shape[0], 1)).astype(np.float32)

            if x_max_neva > x_min_neva:
                h_neva_bs77 = 5.1 * (X_grid - x_min_neva) / (x_max_neva - x_min_neva)
                h_neva_bs77 = np.clip(h_neva_bs77, 0.0, 5.1)
            else:
                h_neva_bs77 = np.zeros_like(X_grid)

            neva_dsm = offset + h_neva_bs77
            arr[neva_mask == 1] = neva_dsm[neva_mask == 1]
            print(f"   Нева: монотонный градиент 0–5.1 м БС-77 применён")
    except Exception as e:
        print(f"   ⚠ Нева: {e}")

    print(f"   Выровнено пикселей воды: {n_flattened:,}")

    # Сохраняем
    arr_out = np.where(np.isnan(arr), -9999, arr).astype(np.float32)
    profile.update(dtype='float32', count=1, nodata=-9999)
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(arr_out, 1)

    del arr, arr_out, water_mask; gc.collect()
    return out_path


# ──────────────────────────────────────────────────────────────────────────────
# 3.2 DTM из дорог с блочной фильтрацией мостов
# ──────────────────────────────────────────────────────────────────────────────
def build_dtm_from_roads(dsm_path, dtm_out_path, crs, res=10,
                          csf_mult=1.0, block_size_m=500):
    """
    DTM из дорожной сети OSM с улучшенной блочной фильтрацией мостов.

    Улучшение относительно v5:
      РАНЬШЕ: Глобальный фильтр (median ± 2σ) — не работает на переменном рельефе.
      ТЕПЕРЬ: Блочный IQR (блоки block_size_m × block_size_m метров).
        Каждый блок: если точка > Q75 + 1.5*IQR — мост/эстакада → исключить.
        Физический смысл: в пределах 500 м рельеф СПб меняется на < 2–3 м.

    csf_mult: множитель сезонной коррекции (1.3 для leaf-on, 1.0 для leaf-off).
    """
    import osmnx as ox
    ox.settings.timeout = 120

    print(f"   CSF-множитель (сезон): ×{csf_mult:.1f}")
    print("   Загрузка дорожной сети из OSM...")

    dsm_10m = os.path.join(WD, "dsm_10m_cov.tif")
    if not os.path.exists(dsm_10m):
        opts = gdal.WarpOptions(format="GTiff", xRes=res, yRes=res,
                                resampleAlg="bilinear", dstNodata=-9999,
                                creationOptions=["COMPRESS=LZW", "TILED=YES"])
        gdal.Warp(dsm_10m, dsm_path, options=opts)

    with rasterio.open(dsm_10m) as src:
        dsm_arr   = src.read(1).astype(np.float32)
        nd        = src.nodata
        if nd is not None:
            dsm_arr[dsm_arr == nd] = np.nan
        transform = src.transform
        profile   = src.profile.copy()
        bounds    = src.bounds

    bbox_gdf = gpd.GeoDataFrame(
        {'geometry': [box(bounds.left, bounds.bottom, bounds.right, bounds.top)]},
        crs=crs
    )
    b = bbox_gdf.to_crs("EPSG:4326").total_bounds

    # Загрузка дорог по полосам широты
    lat_step = 0.08
    all_roads = []
    lat = b[1]
    while lat < b[3]:
        lat_end = min(lat + lat_step, b[3])
        try:
            roads = ox.features_from_bbox(
                bbox=(b[0], lat, b[2], lat_end),
                tags={'highway': ['residential', 'tertiary', 'secondary', 'primary',
                                  'trunk', 'motorway', 'unclassified', 'service',
                                  'living_street', 'pedestrian']}
            )
            if not roads.empty:
                roads = roads[roads.geometry.type.isin(['LineString', 'MultiLineString'])]
                all_roads.append(roads[['geometry']])
                print(f"     {lat:.3f}–{lat_end:.3f}: {len(roads)} дорог")
        except Exception as e:
            print(f"     {lat:.3f}–{lat_end:.3f}: ⚠ {e}")
        lat = lat_end

    if not all_roads:
        print("   ⚠ Дороги не загружены")
        return None

    roads_gdf = gpd.GeoDataFrame(
        pd.concat(all_roads, ignore_index=True),
        crs="EPSG:4326"
    ).to_crs(crs)
    print(f"   Всего дорог: {len(roads_gdf)}")

    # Сэмплирование DSM вдоль дорог
    sample_step = res
    pts_x, pts_y, pts_z = [], [], []

    for geom in roads_gdf.geometry:
        if geom is None or geom.is_empty:
            continue
        lines = [geom] if geom.geom_type == 'LineString' else list(geom.geoms)
        for line in lines:
            dist = 0
            while dist <= line.length:
                pt  = line.interpolate(dist)
                c   = int((pt.x - transform.c) / transform.a)
                r   = int((pt.y - transform.f) / transform.e)
                if 0 <= r < dsm_arr.shape[0] and 0 <= c < dsm_arr.shape[1]:
                    val = dsm_arr[r, c]
                    if not np.isnan(val):
                        pts_x.append(pt.x)
                        pts_y.append(pt.y)
                        pts_z.append(val)
                dist += sample_step

    pts_x = np.array(pts_x)
    pts_y = np.array(pts_y)
    pts_z = np.array(pts_z)
    print(f"   Точек сэмплирования: {len(pts_z):,}")

    # ── УЛУЧШЕНИЕ: БЛОЧНАЯ ФИЛЬТРАЦИЯ МОСТОВ ──
    if len(pts_z) > 100:
        print(f"   Блочная фильтрация мостов (блок {block_size_m} м)...")

        # Разбиваем на блоки по пространственным координатам
        x_min, x_max = pts_x.min(), pts_x.max()
        y_min, y_max = pts_y.min(), pts_y.max()

        n_blocks_x = max(1, int((x_max - x_min) / block_size_m) + 1)
        n_blocks_y = max(1, int((y_max - y_min) / block_size_m) + 1)

        good_mask = np.ones(len(pts_z), dtype=bool)
        n_removed = 0

        for bx in range(n_blocks_x):
            for by in range(n_blocks_y):
                xlo = x_min + bx * block_size_m
                xhi = xlo + block_size_m
                ylo = y_min + by * block_size_m
                yhi = ylo + block_size_m

                block_idx = np.where(
                    (pts_x >= xlo) & (pts_x < xhi) &
                    (pts_y >= ylo) & (pts_y < yhi)
                )[0]

                if len(block_idx) < 5:
                    continue

                zb = pts_z[block_idx]
                q25, q75 = np.percentile(zb, [25, 75])
                iqr = q75 - q25

                if iqr < 0.1:
                    iqr = 0.1  # Минимальный IQR для плоского рельефа

                # Мосты/эстакады: выше Q75 + 1.5*IQR
                # Подземные: ниже Q25 - 3*IQR (более мягкий порог снизу)
                lo = q25 - 3.0 * iqr
                hi = q75 + 1.5 * iqr

                outlier_idx = block_idx[(zb < lo) | (zb > hi)]
                good_mask[outlier_idx] = False
                n_removed += len(outlier_idx)

        pts_x = pts_x[good_mask]
        pts_y = pts_y[good_mask]
        pts_z = pts_z[good_mask]
        print(f"   После блочной фильтрации: {len(pts_z):,} (удалено {n_removed})")

    if len(pts_z) < 1000:
        print("   ⚠ Мало точек для интерполяции")
        return None

    # ── CSF-подобная морфологическая фильтрация ──
    # Имитация Cloth Simulation Filter:
    # Разбиваем на блоки, берём перцентиль (зависит от csf_mult):
    #   leaf_off: P10 (ткань опускается к земле через листву)
    #   leaf_on:  P5  (плотнее листва → нужно «опуститься» ниже)
    csf_percentile = max(2, int(10 / csf_mult))
    print(f"   CSF-перцентиль: P{csf_percentile} (сезонная коррекция ×{csf_mult:.1f})")

    csf_block_m = 50  # Блок CSF: 50 м
    x_min, x_max = pts_x.min(), pts_x.max()
    y_min, y_max = pts_y.min(), pts_y.max()
    nb_x = max(1, int((x_max - x_min) / csf_block_m) + 1)
    nb_y = max(1, int((y_max - y_min) / csf_block_m) + 1)

    pts_z_csf = pts_z.copy()
    for bx in range(nb_x):
        for by in range(nb_y):
            xlo = x_min + bx * csf_block_m
            xhi = xlo + csf_block_m
            ylo = y_min + by * csf_block_m
            yhi = ylo + csf_block_m
            idx = np.where(
                (pts_x >= xlo) & (pts_x < xhi) &
                (pts_y >= ylo) & (pts_y < yhi)
            )[0]
            if len(idx) > 3:
                ground_level = np.percentile(pts_z[idx], csf_percentile)
                # Удаляем точки, которые выше «ткани» (листва, небольшие подъёмы)
                too_high = idx[pts_z[idx] > ground_level + 3.0]
                pts_z_csf[too_high] = ground_level

    # Интерполяция DTM
    print("   Интерполяция DTM...")
    rows_n, cols_n = dsm_arr.shape
    yi = np.arange(rows_n) * transform.e + transform.f + transform.e / 2
    xi = np.arange(cols_n) * transform.a + transform.c + transform.a / 2
    XI, YI = np.meshgrid(xi, yi)

    dtm = griddata((pts_x, pts_y), pts_z_csf, (XI, YI), method='nearest')
    dtm = ndimage.uniform_filter(dtm.astype(np.float32), size=5)

    # DTM не может быть выше DSM (по определению)
    valid_dsm = ~np.isnan(dsm_arr)
    dtm[valid_dsm] = np.minimum(dtm[valid_dsm], dsm_arr[valid_dsm])

    print(f"   DTM: мин={np.nanmin(dtm):.2f}, макс={np.nanmax(dtm):.2f}, "
          f"медиана={np.nanmedian(dtm):.2f} м (WGS84)")

    profile.update(dtype='float32', count=1, nodata=-9999)
    with rasterio.open(dtm_out_path, 'w', **profile) as dst:
        dst.write(dtm.astype(np.float32), 1)

    del dsm_arr, dtm, XI, YI, pts_x, pts_y, pts_z; gc.collect()
    print(f"   DTM сохранён: {dtm_out_path}")
    return dtm_out_path


print("[MODULE 2-3] Coverage mask + DSM→DTM — функции загружены")

In [ ]:
# ======================================================================================
# МОДУЛИ 4–6: МАСКА ЗАТОПЛЕНИЯ + СВЯЗНОСТЬ + КАРТА ВЕРОЯТНОСТИ
# ======================================================================================

def threshold_mask_tiled(dsm_path, out_path, threshold, water_level, tile_sz=4096):
    """
    Тайловая пороговая маска (без изменений относительно v5).
    nodata → заполняется water_level; DSM ≤ threshold → 1.
    """
    src    = gdal.Open(dsm_path)
    cols   = src.RasterXSize
    rows   = src.RasterYSize
    band   = src.GetRasterBand(1)
    src_nd = band.GetNoDataValue()

    drv = gdal.GetDriverByName('GTiff')
    dst = drv.Create(out_path, cols, rows, 1, gdal.GDT_Byte,
                     ['COMPRESS=LZW', 'TILED=YES', 'BLOCKXSIZE=256', 'BLOCKYSIZE=256'])
    dst.SetGeoTransform(src.GetGeoTransform())
    dst.SetProjection(src.GetProjection())
    dst.GetRasterBand(1).SetNoDataValue(255)

    n_ty = (rows + tile_sz - 1) // tile_sz
    for ty in range(n_ty):
        y_off = ty * tile_sz
        y_sz  = min(tile_sz, rows - y_off)
        for tx in range((cols + tile_sz - 1) // tile_sz):
            x_off = tx * tile_sz
            x_sz  = min(tile_sz, cols - x_off)
            tile   = band.ReadAsArray(x_off, y_off, x_sz, y_sz).astype(np.float32)
            nd_mask = np.isnan(tile)
            if src_nd is not None:
                nd_mask |= (tile == src_nd)
            tile[nd_mask] = water_level
            mask = (tile <= threshold).astype(np.uint8)
            dst.GetRasterBand(1).WriteArray(mask, x_off, y_off)
        if (ty + 1) % max(1, n_ty // 5) == 0 or ty == n_ty - 1:
            print(f"     {100*(ty+1)/n_ty:.0f}%")

    dst.FlushCache(); dst = None; src = None


# ──────────────────────────────────────────────────────────────────────────────
# 6.1 Карта вероятности затопления (новое!)
# ──────────────────────────────────────────────────────────────────────────────
def build_probability_flood_map(dsm_path, out_path, offset_ci,
                                 flood_level=3.05, n_samples=20, tile_sz=4096):
    """
    Вычисляет вероятность затопления с учётом неопределённости калибровки.

    Метод Монте-Карло:
      offset ~ Uniform(ci_low, ci_high)
      threshold_i = flood_level + offset_i
      P(flood) = доля симуляций, где DSM <= threshold_i

    Интерпретация карты (0–255):
      0   = P=0%  (никогда не затопляется даже при максимальном offset)
      128 = P=50% (неопределённая зона)
      255 = P=100% (всегда затопляется при любом valide offset)

    Практическая ценность:
      Зоны P=100% → безусловно затопляемые (приоритет эвакуации)
      Зоны P=50–99% → требуют уточнения (дополнительные полевые измерения)
      Зоны P<10% → условно безопасные при данном уровне наводнения
    """
    ci_low  = offset_ci['ci_low']
    ci_high = offset_ci['ci_high']
    offset_samples = np.linspace(ci_low, ci_high, n_samples)
    thresholds     = flood_level + offset_samples

    print(f"   Карта вероятности: {n_samples} симуляций, "
          f"CI=[{ci_low:.3f}, {ci_high:.3f}] м")
    print(f"   Пороги DSM: [{thresholds.min():.3f}, {thresholds.max():.3f}] м")

    src    = gdal.Open(dsm_path)
    cols   = src.RasterXSize
    rows   = src.RasterYSize
    band   = src.GetRasterBand(1)
    src_nd = band.GetNoDataValue()

    drv = gdal.GetDriverByName('GTiff')
    dst = drv.Create(out_path, cols, rows, 1, gdal.GDT_Byte,
                     ['COMPRESS=LZW', 'TILED=YES'])
    dst.SetGeoTransform(src.GetGeoTransform())
    dst.SetProjection(src.GetProjection())
    dst.GetRasterBand(1).SetNoDataValue(255)

    n_ty = (rows + tile_sz - 1) // tile_sz
    for ty in range(n_ty):
        y_off = ty * tile_sz
        y_sz  = min(tile_sz, rows - y_off)
        for tx in range((cols + tile_sz - 1) // tile_sz):
            x_off = tx * tile_sz
            x_sz  = min(tile_sz, cols - x_off)
            tile = band.ReadAsArray(x_off, y_off, x_sz, y_sz).astype(np.float32)
            nd_mask = np.isnan(tile)
            if src_nd is not None:
                nd_mask |= (tile == src_nd)

            # Накапливаем число симуляций, где пиксель затоплен
            count = np.zeros_like(tile, dtype=np.float32)
            for thr in thresholds:
                count += (tile <= thr).astype(np.float32)

            prob = (count / n_samples * 255).astype(np.uint8)
            prob[nd_mask] = 255  # nodata
            dst.GetRasterBand(1).WriteArray(prob, x_off, y_off)

    dst.FlushCache(); dst = None; src = None
    print(f"   Карта вероятности: {out_path}")


def connectivity_analysis(mask_2m, out_conn, seeds, crs,
                           closing_m, conn_res, seed_r,
                           coverage_mask_path=None, dtm_path=None, threshold=None):
    """Связность (без изменений относительно v5)."""
    mask_10m = os.path.join(WD, "mask_10m.tif")
    opts = gdal.WarpOptions(format="GTiff", xRes=conn_res, yRes=conn_res,
                            resampleAlg="max",
                            creationOptions=["COMPRESS=LZW", "TILED=YES"])
    gdal.Warp(mask_10m, mask_2m, options=opts)

    with rasterio.open(mask_10m) as src:
        arr       = src.read(1)
        transform = src.transform
        profile   = src.profile.copy()

    print(f"   Маска 10 м: {arr.shape} ({arr.shape[0]*arr.shape[1]:,} px)")

    if dtm_path and os.path.exists(dtm_path) and threshold is not None:
        with rasterio.open(dtm_path) as dsrc:
            dtm = dsrc.read(1).astype(np.float32)
        dtm[dtm == -9999] = np.nan
        dtm_mask = (dtm <= threshold).astype(np.uint8)
        dtm_mask[np.isnan(dtm)] = 0
        n_dtm = int(np.sum((arr == 0) & (dtm_mask == 1)))
        arr = arr | dtm_mask
        print(f"   + DTM добавил {n_dtm:,} px")
        del dtm, dtm_mask

    closing_px = max(1, int(round(closing_m / conn_res)))
    struct = ndimage.generate_binary_structure(2, 2)
    arr    = ndimage.binary_closing(arr, structure=struct, iterations=closing_px).astype(np.uint8)

    labels, n_feat = ndimage.label(arr)
    print(f"   Компонент: {n_feat}")

    valid_labels = set()
    for lon, lat in seeds:
        pt = gpd.GeoDataFrame({'geometry': [Point(lon, lat)]}, crs="EPSG:4326").to_crs(crs)
        sx, sy = pt.geometry.iloc[0].x, pt.geometry.iloc[0].y
        c  = int((sx - transform.c) / transform.a)
        r  = int((sy - transform.f) / transform.e)
        r0, r1 = max(0, r-seed_r), min(labels.shape[0], r+seed_r+1)
        c0, c1 = max(0, c-seed_r), min(labels.shape[1], c+seed_r+1)
        found = set(np.unique(labels[r0:r1, c0:c1])) - {0}
        if found:
            valid_labels.update(found)
        else:
            print(f"   ⚠ Seed ({lon:.3f},{lat:.3f}) — нет маски в {seed_r*conn_res} м")

    if not valid_labels:
        for lbl in range(1, n_feat + 1):
            if np.sum(labels == lbl) > 5000:
                valid_labels.add(lbl)

    conn = np.isin(labels, list(valid_labels)).astype(np.uint8)

    if coverage_mask_path and os.path.exists(coverage_mask_path):
        with rasterio.open(coverage_mask_path) as csrc:
            cov = csrc.read(1)
        if cov.shape != conn.shape:
            from scipy.ndimage import zoom
            zy = conn.shape[0]/cov.shape[0]; zx = conn.shape[1]/cov.shape[1]
            cov = zoom(cov, (zy, zx), order=0)[:conn.shape[0], :conn.shape[1]]
        conn = conn & cov

    n_final = int(np.sum(conn))
    area_ha = n_final * (conn_res**2) / 10000
    print(f"   Итого: {n_final:,} px (~{area_ha:.0f} га)")

    profile.update(dtype='uint8', nodata=255)
    with rasterio.open(out_conn, 'w', **profile) as dst:
        dst.write(conn, 1)

    del labels, arr, conn; gc.collect()
    return out_conn


def apply_filter_2m(mask_2m, conn_10m, out_final, tile_sz=4096):
    """Upsample фильтр → AND с маской 2 м (без изменений)."""
    conn_2m = os.path.join(WD, "conn_2m.tif")
    src_mask = gdal.Open(mask_2m)
    gt = src_mask.GetGeoTransform()
    opts = gdal.WarpOptions(
        format="GTiff", xRes=abs(gt[1]), yRes=abs(gt[5]),
        resampleAlg="near", creationOptions=["COMPRESS=LZW", "TILED=YES"]
    )
    gdal.Warp(conn_2m, conn_10m, options=opts)
    src_filt = gdal.Open(conn_2m)
    cols, rows = src_mask.RasterXSize, src_mask.RasterYSize
    drv = gdal.GetDriverByName('GTiff')
    dst = drv.Create(out_final, cols, rows, 1, gdal.GDT_Byte,
                     ['COMPRESS=LZW', 'TILED=YES'])
    dst.SetGeoTransform(gt)
    dst.SetProjection(src_mask.GetProjection())
    for ty in range((rows + tile_sz - 1) // tile_sz):
        y_off = ty * tile_sz; y_sz = min(tile_sz, rows - y_off)
        for tx in range((cols + tile_sz - 1) // tile_sz):
            x_off = tx * tile_sz; x_sz = min(tile_sz, cols - x_off)
            m  = src_mask.GetRasterBand(1).ReadAsArray(x_off, y_off, x_sz, y_sz)
            fc = src_filt.RasterXSize; fr = src_filt.RasterYSize
            result = np.zeros_like(m)
            if x_off < fc and y_off < fr:
                fxs = min(x_sz, fc-x_off); fys = min(y_sz, fr-y_off)
                f   = src_filt.GetRasterBand(1).ReadAsArray(x_off, y_off, fxs, fys)
                result[:fys, :fxs] = m[:fys, :fxs] & f
            dst.GetRasterBand(1).WriteArray(result, x_off, y_off)
    dst.FlushCache(); dst = None; src_mask = None; src_filt = None
    if os.path.exists(conn_2m):
        os.remove(conn_2m)


def vectorize_and_map(final_mask, prob_map_path, crs, calib_result, out_html):
    """Векторизация + Folium карта с слоем вероятности."""
    with rasterio.open(final_mask) as src:
        arr       = src.read(1)
        transform = src.transform
        bounds    = src.bounds

    print(f"   Затопленных пикселей: {int(np.sum(arr)):,}")
    if np.sum(arr) == 0:
        print("   Затопления нет."); return None

    geoms = list({'properties': {'v': int(v)}, 'geometry': s}
                 for s, v in shapes(arr, mask=arr == 1, transform=transform))
    del arr; gc.collect()

    if not geoms:
        return None

    gdf = gpd.GeoDataFrame.from_features(geoms, crs=crs)
    del geoms; gc.collect()
    gdf = gdf[gdf.geometry.area >= MIN_AREA]
    gdf = gdf.dissolve()
    gdf.geometry = gdf.geometry.simplify(SIMPLIFY)
    flood_area_ha = gdf.geometry.area.sum() / 10000
    print(f"   Площадь затопления: {flood_area_ha:.0f} га")

    bbox_wgs = gpd.GeoDataFrame(
        {'geometry': [box(bounds.left, bounds.bottom, bounds.right, bounds.top)]},
        crs=crs
    ).to_crs("EPSG:4326").total_bounds

    cy, cx = (bbox_wgs[1]+bbox_wgs[3])/2, (bbox_wgs[0]+bbox_wgs[2])/2
    m = folium.Map(location=[cy, cx], zoom_start=11, tiles='CartoDB positron')

    # Слой 1: Бинарная зона затопления
    flood_wgs = gdf.to_crs("EPSG:4326")
    folium.GeoJson(
        flood_wgs, name='Зона затопления (P=100%)',
        style_function=lambda x: {'fillColor': '#1E90FF', 'color': '#0000CD',
                                   'weight': 0.5, 'fillOpacity': 0.6}
    ).add_to(m)

    # Информационная панель
    offset  = calib_result['offset']
    ci_low  = calib_result['ci_low']
    ci_high = calib_result['ci_high']
    note    = calib_result['note']

    info_html = f"""
    <div style="position:fixed; bottom:30px; left:30px; z-index:1000;
                background:white; padding:12px; border-radius:8px;
                box-shadow:2px 2px 8px rgba(0,0,0,0.3); font-size:12px;">
        <b>FLOOD MAP v6</b><br>
        Уровень: {FLOOD_LEVEL} м БС-77<br>
        Дата стрипов: {STRIP_DATE}<br>
        Offset: {offset:.3f} м [{note}]<br>
        CI 95%: [{ci_low:.3f}, {ci_high:.3f}] м<br>
        Площадь: <b>{flood_area_ha:.0f} га</b><br>
        δ_BS77 = {DELTA_BS77:+.2f} м (Кронштадт)
    </div>
    """
    m.get_root().html.add_child(folium.Element(info_html))

    # Landuse impact
    try:
        import osmnx as ox
        ox.settings.timeout = 60
        fb = flood_wgs.total_bounds
        lu = ox.features_from_bbox(bbox=(fb[0],fb[1],fb[2],fb[3]), tags={'landuse': True})
        if not lu.empty:
            lu = lu[lu.geometry.type.isin(['Polygon','MultiPolygon'])].to_crs(crs)
            impact = gpd.overlay(lu, gdf, how='intersection')
            impact['ha'] = impact.geometry.area / 10000
            print("\n   Landuse в зоне затопления:")
            print(impact.groupby('landuse')['ha'].sum().sort_values(ascending=False).head(10).to_string())
            impact_wgs = impact.to_crs("EPSG:4326")
            folium.GeoJson(
                impact_wgs, name='Инфраструктура',
                style_function=lambda x: {'fillColor': '#FF4500', 'color': 'black',
                                           'weight': 0.5, 'fillOpacity': 0.5},
                tooltip=folium.GeoJsonTooltip(
                    fields=['landuse','ha'], aliases=['Тип','Га'], localize=True
                )
            ).add_to(m)
    except Exception as e:
        print(f"   ⚠ OSM landuse: {e}")

    folium.LayerControl().add_to(m)
    m.save(out_html)
    print(f"   Карта: {out_html}")
    return m


print("[MODULE 4-6] Flood mask + Connectivity + Probability map — функции загружены")

In [ ]:
# ======================================================================================
# MAIN — ПОЛНЫЙ ПАЙПЛАЙН v6
# ======================================================================================

def main():
    print("=" * 70)
    print(" FLOOD MAP v6 — ГЕОДЕЗИЧЕСКИ СТРОГИЙ ПАЙПЛАЙН")
    print(f" Уровень: {FLOOD_LEVEL} м БС-77  |  Дата: {STRIP_DATE}")
    print(f" δ_BS77 = {DELTA_BS77:+.2f} м  |  N_EGM(центр) = {EGM_POLY_COEFFS['C00']:.2f} м")
    print("=" * 70)

    # Пути к файлам
    dsm_coregistered = os.path.join(WD, "dsm_coregistered.tif")
    dsm_flattened    = os.path.join(WD, "dsm_water_flat.tif")
    stable_mask      = os.path.join(WD, "stable_surface_mask.tif")
    cov_mask         = os.path.join(WD, "coverage_mask.tif")
    dtm_10m          = os.path.join(WD, "dtm_roads_10m.tif")
    offset_map       = os.path.join(WD, "offset_map.tif")
    mask_2m          = os.path.join(WD, "mask_2m.tif")
    conn_10m         = os.path.join(WD, "conn_10m.tif")
    final_2m         = os.path.join(WD, "flood_final_2m.tif")
    prob_map         = os.path.join(WD, "flood_probability.tif")
    out_html         = os.path.join(WD, "spb_flood_v6.html")
    calib_json       = os.path.join(WD, "calibration_metadata.json")

    # ──────────────────────────────────────────────────────────────────────────
    # ШАГ 0: Корегистрация по ICESat-2 ATL08 (Nuth & Kääb)
    # ──────────────────────────────────────────────────────────────────────────
    print(f"\n[0/8] ICESat-2 ATL08 → 3D корегистрация (Nuth & Kääb)...")

    # Загружаем ATL08
    atl08_files = download_atl08_for_region(
        lat_min=59.5, lat_max=60.4,
        lon_min=28.5, lon_max=31.5,
        date_start="2018-10-14",
        date_end=STRIP_DATE  # Только треки до/около даты съёмки стрипа
    )

    shift_result = {'dx': 0.0, 'dy': 0.0, 'dz': 0.0,
                    'rmse_before': np.nan, 'rmse_after': np.nan, 'n_points': 0}

    if atl08_files:
        # Маска стабильных поверхностей
        create_stable_surface_mask(INPUT_DSM, CRS_METRIC, out_path=stable_mask)

        # Загружаем и фильтруем ATL08
        atl08_gdf = load_all_atl08(atl08_files, CRS_METRIC)

        if not atl08_gdf.empty:
            # Корегистрация
            shift_result = nuth_kaab_coregistration(
                INPUT_DSM, atl08_gdf, stable_mask, CRS_METRIC
            )

    # Применяем 3D-сдвиг
    dsm_work = apply_3d_shift_to_dsm(
        INPUT_DSM, shift_result, dsm_coregistered, CRS_METRIC
    )

    print(f"   Вектор сдвига: dx={shift_result['dx']:+.3f}, "
          f"dy={shift_result['dy']:+.3f}, dz={shift_result['dz']:+.3f} м")

    # ──────────────────────────────────────────────────────────────────────────
    # ШАГ 1: Вертикальная калибровка
    # ──────────────────────────────────────────────────────────────────────────
    print(f"\n[1/8] Вертикальная калибровка...")
    calib_result = calibrate(dsm_work, CONTROL_POINTS, GAUGE_CSV, CRS_METRIC)

    offset      = calib_result['offset']
    threshold   = FLOOD_LEVEL + offset
    water_level = offset  # 0 м БС-77 в DSM-координатах

    print(f"\n   ═════════════════════════════════════════")
    print(f"   ИТОГ КАЛИБРОВКИ:")
    print(f"   offset          = {offset:.4f} м")
    print(f"   threshold (DSM) = {threshold:.4f} м")
    print(f"   95% CI:           [{calib_result['ci_low']:.3f}, {calib_result['ci_high']:.3f}] м")
    print(f"   ═════════════════════════════════════════")

    # ──────────────────────────────────────────────────────────────────────────
    # ШАГ 1b: Пространственно-переменный offset
    # ──────────────────────────────────────────────────────────────────────────
    print(f"\n[1b/8] Карта пространственного offset...")
    build_spatial_offset_map(dsm_work, calib_result, CRS_METRIC)

    # ──────────────────────────────────────────────────────────────────────────
    # ШАГ 2: Маска покрытия
    # ──────────────────────────────────────────────────────────────────────────
    print(f"\n[2/8] Маска покрытия...")
    create_coverage_mask(dsm_work, cov_mask, CRS_METRIC)

    # ──────────────────────────────────────────────────────────────────────────
    # ШАГ 3a: Water Flattening
    # ──────────────────────────────────────────────────────────────────────────
    print(f"\n[3a/8] Water Flattening (Финский залив, Нева, водоёмы)...")
    dsm_work = apply_water_flattening(
        dsm_work, dsm_flattened, CRS_METRIC, offset
    )

    # ──────────────────────────────────────────────────────────────────────────
    # ШАГ 3b: DSM → DTM (дороги + CSF + блочная фильтрация мостов)
    # ──────────────────────────────────────────────────────────────────────────
    print(f"\n[3b/8] DSM → DTM (мосты: блочный IQR, сезон: {get_leaf_status(STRIP_DATE)[0]})...")
    csf_mult = calib_result.get('csf_mult', 1.0)
    try:
        build_dtm_from_roads(dsm_work, dtm_10m, CRS_METRIC,
                             csf_mult=csf_mult, block_size_m=500)
    except Exception as e:
        print(f"   ⚠ DTM не построен: {e}")
        dtm_10m = None

    # ──────────────────────────────────────────────────────────────────────────
    # ШАГ 4: Бинарная маска 2 м
    # ──────────────────────────────────────────────────────────────────────────
    print(f"\n[4/8] Пороговая маска 2 м (threshold={threshold:.2f} м DSM)...")
    threshold_mask_tiled(dsm_work, mask_2m, threshold, water_level)

    # ──────────────────────────────────────────────────────────────────────────
    # ШАГ 5: Связность
    # ──────────────────────────────────────────────────────────────────────────
    print(f"\n[5/8] Связность + DTM-коррекция...")
    connectivity_analysis(
        mask_2m, conn_10m, WATER_SEEDS, CRS_METRIC,
        CLOSING_M, CONN_RES, SEED_R,
        coverage_mask_path=cov_mask,
        dtm_path=dtm_10m if dtm_10m and os.path.exists(dtm_10m) else None,
        threshold=threshold
    )

    # ──────────────────────────────────────────────────────────────────────────
    # ШАГ 6: Карта вероятности затопления (новая)
    # ──────────────────────────────────────────────────────────────────────────
    print(f"\n[6/8] Карта вероятности затопления (CI Monte-Carlo)...")
    build_probability_flood_map(
        dsm_work, prob_map,
        offset_ci=calib_result,
        flood_level=FLOOD_LEVEL,
        n_samples=50
    )

    # ──────────────────────────────────────────────────────────────────────────
    # ШАГ 7: Финальная 2 м маска
    # ──────────────────────────────────────────────────────────────────────────
    print(f"\n[7/8] Финальная маска 2 м...")
    apply_filter_2m(mask_2m, conn_10m, final_2m)
    for f in [mask_2m, conn_10m]:
        if os.path.exists(f):
            os.remove(f)

    # ──────────────────────────────────────────────────────────────────────────
    # ШАГ 8: Векторизация + Folium карта
    # ──────────────────────────────────────────────────────────────────────────
    print(f"\n[8/8] Карта и метаданные...")
    vectorize_and_map(final_2m, prob_map, CRS_METRIC, calib_result, out_html)

    # Экспорт метаданных калибровки в JSON
    metadata = {
        "version": "v6",
        "strip_date": STRIP_DATE,
        "flood_level_bs77_m": FLOOD_LEVEL,
        "calibration": {
            "offset_m": offset,
            "ci_95_low": calib_result['ci_low'],
            "ci_95_high": calib_result['ci_high'],
            "source": calib_result['note'],
        },
        "coregistration": {
            "dx_m": shift_result['dx'],
            "dy_m": shift_result['dy'],
            "dz_m": shift_result['dz'],
            "rmse_before_m": shift_result['rmse_before'],
            "rmse_after_m":  shift_result['rmse_after'],
            "n_icesat2_points": shift_result['n_points'],
        },
        "geoid": {
            "N_egm2008_center_m": EGM_POLY_COEFFS['C00'],
            "delta_bs77_m": DELTA_BS77,
            "delta_bs77_sigma_m": DELTA_BS77_SIGMA,
            "reference": "GPS-нивелирование GEOBRIDGE, 12 точек ЛО"
        },
        "threshold_dsm_m": threshold,
        "leaf_status": get_leaf_status(STRIP_DATE)[0],
        "csf_multiplier": calib_result.get('csf_mult', 1.0),
    }

    with open(calib_json, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f"\n{'='*70}")
    print(f" ГОТОВО! Flood Map v6")
    print(f" Карта:        {out_html}")
    print(f" Маска (2 м):  {final_2m}")
    print(f" Вероятность:  {prob_map}")
    print(f" Метаданные:   {calib_json}")
    print(f"{'='*70}")


main()

In [ ]:
# ======================================================================================
# ЯЧЕЙКА 3 (УЛУЧШЕНА): КРОСС-СВЕРКА + ДИАГНОСТИКА КАЛИБРОВКИ
# ======================================================================================
#
# Запускается независимо от main() для верификации offset.
# HARD_POINTS: только открытые поверхности (ВПП, стадионы, дамбы).
# НИКОГДА: набережные, вокзалы, узкие улицы.
# ======================================================================================

import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from osgeo import gdal
import os

# ── Параметры ──
STRIP_MSK_HOUR_START = 15
STRIP_MSK_HOUR_END   = 16

# Работаем с откорегистрованным DSM (если есть) или исходным
dem_path = os.path.join(WD, "dsm_coregistered.tif")
if not os.path.exists(dem_path):
    dem_path = INPUT_DSM
    print(f"⚠ Корегистрованный DSM не найден, используем: {dem_path}")
else:
    print(f"✓ Используем откорегистрованный DSM: {dem_path}")


def extract_dem_at_point(dem_arr, transform, lon, lat, crs, radius_px=5):
    pt  = gpd.GeoDataFrame({'geometry': [Point(lon, lat)]}, crs="EPSG:4326").to_crs(crs)
    px  = pt.geometry.iloc[0].x
    py  = pt.geometry.iloc[0].y
    col = int((px - transform.c) / transform.a)
    row = int((py - transform.f) / transform.e)
    r0  = max(0, row - radius_px);  r1 = min(dem_arr.shape[0], row + radius_px + 1)
    c0  = max(0, col - radius_px);  c1 = min(dem_arr.shape[1], col + radius_px + 1)
    patch = dem_arr[r0:r1, c0:c1]
    valid = patch[~np.isnan(patch) & (patch > -5) & (patch < 300)]
    return (float(np.median(valid)), row, col) if len(valid) > 0 else (np.nan, row, col)


def get_water_level_at_time(df, col, date_str, h_start, h_end):
    try:
        day = df[df['_dt'].dt.strftime('%Y-%m-%d') == date_str]
        win = day[(day['_dt'].dt.hour >= h_start) & (day['_dt'].dt.hour <= h_end)]
        v   = pd.to_numeric(win[col], errors='coerce').dropna() / 100
        return (round(float(v.mean()), 3), len(v)) if not v.empty else (np.nan, 0)
    except Exception:
        return np.nan, 0


# ── Загружаем DEM (10 м для скорости) ──
print("\n[1] Загрузка DEM...")
import rasterio
from osgeo import gdal

dem_10m_check = os.path.join(WD, "dem_10m_check.tif")
if not os.path.exists(dem_10m_check):
    opts = gdal.WarpOptions(format="GTiff", xRes=10, yRes=10,
                            resampleAlg="bilinear", dstNodata=-9999,
                            creationOptions=["COMPRESS=LZW"])
    gdal.Warp(dem_10m_check, dem_path, options=opts)

with rasterio.open(dem_10m_check) as src:
    dem_arr   = src.read(1).astype(float)
    nd        = src.nodata
    transform = src.transform
    if nd is not None:
        dem_arr[dem_arr == nd] = np.nan

print(f"   DEM: {dem_arr.shape}, диапазон [{np.nanmin(dem_arr):.1f}, {np.nanmax(dem_arr):.1f}] м")

# ── Загружаем CSV с постами ──
print("\n[2] Загрузка данных водомерных постов...")
GAUGES = [
    {"name": "Горный институт", "col": "ГИ",    "lat": 59.9284, "lon": 30.2701, "primary": True},
    {"name": "Финский залив С1", "col": "С1 ФЗ", "lat": 59.9903, "lon": 29.6917, "primary": False},
    {"name": "Финский залив С2", "col": "С2 ФЗ", "lat": 59.9903, "lon": 29.6917, "primary": False},
]

df = None
try:
    df = pd.read_csv(GAUGE_CSV)
    df['_dt'] = pd.to_datetime(df.iloc[:, 0])
    print(f"   CSV: {len(df)} строк, колонки: {list(df.columns[:6])}")
except Exception as e:
    print(f"   ⚠ CSV не загружен: {e}")

# ── Вычисляем GEOID_OFFSET по водомерным постам ──
print(f"\n[3] Калибровка по водомерным постам ({STRIP_DATE}, МСК {STRIP_MSK_HOUR_START}–{STRIP_MSK_HOUR_END}ч)")
print(f"{'Пост':<25s} {'DEM(WGS84)':>10s} {'Уровень':>10s} {'N_eff(offset)':>14s} {'n':>4s}")
print("-" * 70)

geoid_estimates = []

for gauge in GAUGES:
    dem_val, row, col = extract_dem_at_point(
        dem_arr, transform, gauge["lon"], gauge["lat"], CRS_METRIC
    )
    wl, n = (np.nan, 0) if df is None else get_water_level_at_time(
        df, gauge["col"], STRIP_DATE, STRIP_MSK_HOUR_START, STRIP_MSK_HOUR_END
    )

    if np.isnan(dem_val) or np.isnan(wl):
        print(f"{gauge['name']:<25s} {'N/A':>10s} {wl if not np.isnan(wl) else 'N/A':>10s}")
        continue

    geoid = dem_val - wl  # N_effective = DSM - H_BS77_water
    star  = " ★" if gauge["primary"] else ""
    # Теоретическое: N_EGM + delta_BS77
    n_egm_pt = egm2008_undulation(gauge['lat'], gauge['lon'])
    theory   = n_egm_pt + DELTA_BS77
    diff     = geoid - theory
    print(f"{gauge['name']:<25s} {dem_val:>10.2f} {wl:>10.2f} {geoid:>12.2f} {n:>4d}{star}  "
          f"(теория={theory:.2f}, Δ={diff:+.3f})")
    geoid_estimates.append({'name': gauge['name'], 'geoid': geoid, 'primary': gauge['primary']})

GEOID_OFFSET = SPB_OFFSET_FALLBACK
if geoid_estimates:
    primary = [g for g in geoid_estimates if g['primary']]
    if primary:
        GEOID_OFFSET = primary[0]['geoid']
        print(f"\n  ★ Принятый GEOID_OFFSET (по ГИ): {GEOID_OFFSET:.4f} м")
    else:
        GEOID_OFFSET = float(np.median([g['geoid'] for g in geoid_estimates]))
        print(f"\n  Медианный GEOID_OFFSET: {GEOID_OFFSET:.4f} м")

    if len(geoid_estimates) > 1:
        geoids = [g['geoid'] for g in geoid_estimates]
        spread = max(geoids) - min(geoids)
        warn = " ⚠" if spread > 0.5 else ""
        print(f"  Разброс: {spread:.3f} м (≤ 0.5 м норма){warn}")
        if spread > 0.5:
            print("  → Разные стрипы ArcticDEM → тайловая калибровка")


# ── Кросс-сверка по HARD_POINTS ──
print(f"\n\n[4] Кросс-сверка по твёрдым точкам (открытый рельеф)")
print("-" * 85)
print(f"{'Точка':<38s} {'H_ort(DEM)':>10s} {'Ожид.':>7s} {'Δ':>7s} Источник")
print(f"  (H_ort = DEM_WGS84 - GEOID_OFFSET; |Δ| < {TOLERANCE_GOOD} м = отлично)")
print("-" * 85)

errors = []
for name, lat, lon, known_elev, source in HARD_POINTS:
    dem_val, _, _ = extract_dem_at_point(dem_arr, transform, lon, lat, CRS_METRIC)
    if np.isnan(dem_val):
        print(f"{name:<38s} {'N/A':>10s} {known_elev:>7.1f} {'N/A':>7s} {source}")
        continue

    dem_ortho = dem_val - GEOID_OFFSET
    delta     = dem_ortho - known_elev
    errors.append(delta)

    ok = "✓" if abs(delta) < TOLERANCE_GOOD else ("≈" if abs(delta) < TOLERANCE_OK else "⚠")
    diag = " (крыша?)" if delta > 10 else (" (вода?)" if delta < -5 else "")
    print(f"{name:<38s} {dem_ortho:>10.2f} {known_elev:>7.1f} {delta:>+7.2f} {source} {ok}{diag}")

if errors:
    errors = np.array(errors)
    rms    = np.sqrt(np.mean(errors**2))
    bias   = np.mean(errors)
    print("-" * 85)
    print(f"  Среднее:  {bias:>+.3f} м  |  Стд.: {np.std(errors):.3f} м  |  RMS: {rms:.3f} м")

    if rms < TOLERANCE_GOOD:
        quality = f"ОТЛИЧНО (RMS < {TOLERANCE_GOOD} м)"
    elif rms < TOLERANCE_OK:
        quality = f"ХОРОШО (RMS < {TOLERANCE_OK} м — учтён DSM-эффект)"
    elif rms < 10:
        quality = "ПРИЕМЛЕМО — проверьте координаты"
    else:
        quality = "⚠ ПЛОХО — точки попали в здания или воду"

    print(f"  Качество: {quality}")

    if abs(bias) > TOLERANCE_GOOD:
        corrected = GEOID_OFFSET + bias
        print(f"\n  Систематический сдвиг: {bias:+.3f} м")
        print(f"  Уточнённый offset: {GEOID_OFFSET:.3f} + {bias:.3f} = {corrected:.3f} м")
        print(f"  (применяется только для истинного DTM без зданий)")


# ── Итог ──
print(f"\n\n{'='*70}")
print(f" ПАРАМЕТРЫ ДЛЯ ОСНОВНОГО СКРИПТА (Flood Map v6)")
print(f"{'='*70}")
print(f"""
  GEOID_OFFSET = {GEOID_OFFSET:.4f}  # м  (эмпирический, по посту ГИ)
  DELTA_BS77   = {DELTA_BS77:+.4f}  # м  (теоретическая поправка Кронштадт)
  N_EGM(центр)  = {EGM_POLY_COEFFS['C00']:.4f}  # м  (ундуляция EGM2008)

  Порог затопления (DSM):
    threshold = FLOOD_LEVEL + GEOID_OFFSET = {FLOOD_LEVEL} + {GEOID_OFFSET:.4f} = {FLOOD_LEVEL + GEOID_OFFSET:.4f} м

  Формула:
    H_BS77 = DSM_WGS84 - {GEOID_OFFSET:.4f}

  Корегистрация ATL08:
    → Запустите ячейку [cell-icesat2] с данными NASA Earthdata
    → Логин: https://urs.earthdata.nasa.gov/

  Сезон: {get_leaf_status(STRIP_DATE)[0]} (CSF-множитель ×{get_leaf_status(STRIP_DATE)[1]:.1f})
""")